In [1]:
!pip install langchain langchain-ollama langgraph openai faiss-cpu sqlalchemy fastapi uvicorn

In [2]:
from typing import TypedDict

class AgentState(TypedDict):
    query: str
    route: str
    result: str

In [3]:
# Web Search Tool
from langchain.tools import tool
import requests

@tool(description="Search the web for a query and return summarized results.")
def web_search(query: str) -> str:
    """Search the web for a query and return summarized results."""
    # Replace with SerpAPI / Tavily
    return f"Web results for: {query}"


# SQL Tool
from sqlalchemy import create_engine, text

engine = create_engine("sqlite:///example.db")

@tool(description="Execute a SQL query against a local SQLite database and return the results.")
def sql_query(query: str) -> str:
    """Execute a SQL query against a local SQLite database and return the results."""
    with engine.connect() as conn:
        result = conn.execute(text(query))
        return str(result.fetchall())

In [4]:
def router(state: AgentState):
    q = state["query"].lower()

    if "database" in q or "sql" in q:
        return {"route": "sql"}
    elif "latest" in q or "news" in q:
        return {"route": "web"}
    else:
        return {"route": "rag"}

In [5]:
from langchain_ollama.chat_models import ChatOllama
from langchain_ollama.embeddings import OllamaEmbeddings

llm = ChatOllama(
    model="smollm:latest",
    validate_model_on_init=True,
    temperature=0.2,
    num_predict=512,
)

def web_agent(state: AgentState):
    result = web_search.invoke(state["query"])
    return {"result": result}


def sql_agent(state: AgentState):
    # LLM generates SQL (simplified)
    sql = "SELECT * FROM users LIMIT 5;"
    result = sql_query.invoke(sql)
    return {"result": result}
 

In [8]:
# RAG Agent
from pathlib import Path
from langchain_core.documents.base import Document
from langchain_community.vectorstores import FAISS

embeddings = OllamaEmbeddings(
    model="nomic-embed-text:latest",
    validate_model_on_init=True,
)

index_path = Path("faiss_index")
if index_path.exists() and (index_path / "index.faiss").exists():
    vectorstore = FAISS.load_local(
        "faiss_index",
        embeddings,
        allow_dangerous_deserialization=True,
    )
else:
    sample_docs = [
        Document(
            page_content="This is a sample document for vector search. Replace this with your own data.",
            metadata={"source": "example"},
        ),
        Document(
            page_content="Another example paragraph for the local FAISS index.",
            metadata={"source": "example"},
        ),
    ]
    vectorstore = FAISS.from_documents(sample_docs, embeddings)
    vectorstore.save_local("faiss_index")


def rag_agent(state: AgentState):
    docs = vectorstore.similarity_search(state["query"])
    context = "\n".join([d.page_content for d in docs])

    prompt = f"""
Answer using context:
{context}

Question: {state['query']}
"""

    response = llm.invoke(prompt)
    return {"result": response.content}

In [9]:
from langgraph.graph import StateGraph, END

graph = StateGraph(AgentState)

graph.add_node("router", router)
graph.add_node("web", web_agent)
graph.add_node("sql", sql_agent)
graph.add_node("rag", rag_agent)

# Routing Logic
def route_decision(state: AgentState):
    return state["route"]

graph.add_conditional_edges(
    "router",
    route_decision,
    {
        "web": "web",
        "sql": "sql",
        "rag": "rag"
    }
)

graph.add_edge("web", END)
graph.add_edge("sql", END)
graph.add_edge("rag", END)

graph.set_entry_point("router")

app = graph.compile()

In [10]:
from fastapi import FastAPI

api = FastAPI()

@api.post("/query")
def query_api(payload: dict):
    result = app.invoke({"query": payload["query"]})
    return {"response": result["result"]}

In [11]:
def guardrails(response: str):
    if "password" in response.lower():
        return "Restricted content"
    return response